# Cargo Network Optimization — PuLP Linear Programming

This notebook contains the full LP formulation, constraint definitions, solver call, and constraint verification logic for a multi-node cargo transshipment network.

**Results from the full coursework dataset:**
- Total minimum cost: **$200,863.75**
- Solver status: Optimal
- CVG fully utilized at 95,650 tons; AFW carried 38,097 of 44,350 available tons
- All hub capacity, focus-city capacity, demand satisfaction, and flow-balance constraints verified

*The raw dataset is not included in this repository. Output cells have been cleared. The notebook is published to demonstrate the formulation and methodology.*

In [ ]:
from pulp import LpProblem, LpVariable, LpMinimize, LpStatus, lpSum, PULP_CBC_CMD
import pandas as pd
from pathlib import Path

# =========================
# BRN2 Task 3 (WGU) Notebook
# Style intentionally matches the "all-code-cells + comments" example you shared.
# =========================

DATA_DIR = Path("data")          # put your CSVs here (same folder as this notebook)
OUT_DIR  = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

# ---- Option A: Load from CSVs (recommended) ----
# Expected files (edit these names if yours differ):
#   hubs.csv               columns: hub, capacity   (optional: current_tons)
#   focus_cities.csv       columns: focus_city, capacity (optional: airport)
#   centers.csv            columns: country, center, demand   (or just center, demand)
#   costs.csv              columns: src, dest, cost
#
# If you *don't* have CSVs, scroll down to Option B and paste dictionaries.

def try_load_csvs():
    hubs_fp = DATA_DIR / "hubs.csv"
    foc_fp  = DATA_DIR / "focus_cities.csv"
    cen_fp  = DATA_DIR / "centers.csv"
    cost_fp = DATA_DIR / "costs.csv"
    if not (hubs_fp.exists() and foc_fp.exists() and cen_fp.exists() and cost_fp.exists()):
        return None

    hubs_df = pd.read_csv(hubs_fp)
    foc_df  = pd.read_csv(foc_fp)
    cen_df  = pd.read_csv(cen_fp)
    cost_df = pd.read_csv(cost_fp)

    # hubs: {"CVG":{"capacity":95650, "current_tons":...}, ...}
    hubs = {}
    for _, r in hubs_df.iterrows():
        hub = str(r["hub"])
        hubs[hub] = {"capacity": float(r["capacity"])}
        if "current_tons" in hubs_df.columns:
            hubs[hub]["current_tons"] = float(r["current_tons"])

    # focus cities: {"Leipzig":{"capacity":85000, "airport":...}, ...}
    focus_cities = {}
    for _, r in foc_df.iterrows():
        f = str(r["focus_city"])
        focus_cities[f] = {"capacity": float(r["capacity"])}
        if "airport" in foc_df.columns:
            focus_cities[f]["airport"] = str(r["airport"])

    # centers: nested dict by country to match the sample structure
    centers = {}
    if "country" in cen_df.columns:
        for _, r in cen_df.iterrows():
            country = str(r["country"])
            city    = str(r["center"])
            demand  = float(r["demand"])
            centers.setdefault(country, {})[city] = demand
    else:
        # if no country column, put all centers under a single key
        for _, r in cen_df.iterrows():
            city   = str(r["center"])
            demand = float(r["demand"])
            centers.setdefault("All", {})[city] = demand

    # costs: nested dict cost[src][dest] = cost
    cost = {}
    for _, r in cost_df.iterrows():
        src  = str(r["src"])
        dest = str(r["dest"])
        c    = float(r["cost"])
        cost.setdefault(src, {})[dest] = c

    return hubs, focus_cities, centers, cost

loaded = try_load_csvs()

# ---- Option B: Paste dictionaries (if not using CSVs) ----
if loaded is None:
    # Defining Hub cities (name, current tons, capacity)
    hubs = {
        "CVG": {"current_tons": 82800, "capacity": 95650},
        "AFW": {"current_tons": 38400, "capacity": 44350},
    }

    # Defining Focus Cities (name, airport, capacity)
    focus_cities = {
        "Leipzig": {"airport": "Leipzig/Halle Airport", "capacity": 85000},
        "Hyderabad": {"airport": "Rajiv Gandhi International Airport", "capacity": 19000},
        "San Bernardino": {"airport": "San Bernardino International Airport", "capacity": 36000},
    }

    # Defining centers (country -> city -> demand)
    # NOTE: Paste your full center list here if you're not loading from CSV.
    centers = {
        "France": {"Paris": 6500},
        "Germany": {"Cologne": 640, "Hanover": 180},
        "India": {"Bangalore": 9100, "Coimbatore": 570, "Delhi": 19000, "Mumbai": 14800},
        # ... (continue your full list)
    }

    # Defining ALL costs (hub->focus and focus->center) in ONE nested dict:
    #   cost[src][dest] = dollars_per_ton (or whatever units you use)
    # NOTE: Paste or load your full cost matrix here.
    cost = {
        "CVG": {"Leipzig": 1.0, "Hyderabad": 2.0, "San Bernardino": 3.0},
        "AFW": {"Leipzig": 1.2, "Hyderabad": 2.2, "San Bernardino": 3.2},
        "Leipzig": {"Paris": 4.0, "Cologne": 5.0},
        "Hyderabad": {"Delhi": 6.0, "Mumbai": 7.0},
        "San Bernardino": {"Paris": 8.0},
    }
else:
    hubs, focus_cities, centers, cost = loaded

print("Loaded hubs:", list(hubs.keys()))
print("Loaded focus cities:", list(focus_cities.keys()))
print("Loaded countries:", list(centers.keys())[:5], "...")


In [ ]:
# Flattening the centers (country->city->demand) into city->demand
def flatten_centers(centers_dict):
    return {city: demand for country, cities in centers_dict.items() for city, demand in cities.items()}

flattened_centers = flatten_centers(centers)

# Default cost (high number for unconnected cities so they cannot be seen as efficient)
DEFAULT_COST = 10000

# Fill all missing costs with default value 10000
def get_cost(src, dest, cost_data, default_cost=DEFAULT_COST):
    return cost_data.get(src, {}).get(dest, default_cost)

print("Total centers:", len(flattened_centers))


In [ ]:
# Defining the cost between hubs, focus cities, and centers
# (In this notebook, we keep everything inside `cost[src][dest]` and call get_cost() for safety.)

# Quick sanity checks (optional)
# - If you see lots of DEFAULT_COST being used, it usually means you are missing cost rows.
sample_checks = [
    (list(hubs.keys())[0], list(focus_cities.keys())[0]),
    (list(focus_cities.keys())[0], list(flattened_centers.keys())[0]),
]
for src, dest in sample_checks:
    print(f"Cost {src} -> {dest} =", get_cost(src, dest, cost))

# If you want, you can compute what % of lanes are missing:
missing_hf = sum(1 for h in hubs for f in focus_cities if get_cost(h,f,cost)==DEFAULT_COST)
missing_fc = sum(1 for f in focus_cities for c in flattened_centers if get_cost(f,c,cost)==DEFAULT_COST)
total_hf   = len(hubs)*len(focus_cities)
total_fc   = len(focus_cities)*len(flattened_centers)
print(f"Missing hub->focus lanes: {missing_hf}/{total_hf}")
print(f"Missing focus->center lanes: {missing_fc}/{total_fc}")


In [ ]:
# Defining the optimization and variables
prob = LpProblem("Cargo_Optimization", LpMinimize)

# Decision variables:
# x[h][f] = shipment from hub h to focus city f
# y[f][c] = shipment from focus city f to center c
x = LpVariable.dicts("shipment_from_hub_to_focus", (hubs.keys(), focus_cities.keys()), lowBound=0, cat="Continuous")
y = LpVariable.dicts("shipment_from_focus_to_center", (focus_cities.keys(), flattened_centers.keys()), lowBound=0, cat="Continuous")

# Objective function: minimize total shipping cost
prob += (
    lpSum(get_cost(h, f, cost) * x[h][f] for h in hubs for f in focus_cities)
    + lpSum(get_cost(f, c, cost) * y[f][c] for f in focus_cities for c in flattened_centers)
), "Total_Shipping_Cost"


In [ ]:
# Defining a list of constraints

# 1) Hub capacity constraints
for h in hubs:
    prob += lpSum(x[h][f] for f in focus_cities) <= hubs[h]["capacity"], f"HubCapacity_{h}"

# 2) Focus city processing capacity
for f in focus_cities:
    prob += lpSum(y[f][c] for c in flattened_centers) <= focus_cities[f]["capacity"], f"FocusCapacity_{f}"

# 3) Demand satisfaction at each center
for c, demand in flattened_centers.items():
    prob += lpSum(y[f][c] for f in focus_cities) == demand, f"CenterDemand_{c}"

# 4) Flow balance at each focus city (what flows in must flow out)
for f in focus_cities:
    prob += lpSum(x[h][f] for h in hubs) == lpSum(y[f][c] for c in flattened_centers), f"FlowBalance_{f}"

# Solve
solver = PULP_CBC_CMD(msg=False)
result_status = prob.solve(solver)

print("Solver Status:", LpStatus[prob.status])
print("Objective Value (Total Cost):", prob.objective.value())

# Show only non-zero shipments
shipments = []

for h in hubs:
    for f in focus_cities:
        val = x[h][f].varValue
        if val is not None and val > 1e-6:
            shipments.append({"src": h, "dest": f, "tons": val, "lane": "hub_to_focus", "unit_cost": get_cost(h,f,cost)})

for f in focus_cities:
    for c in flattened_centers:
        val = y[f][c].varValue
        if val is not None and val > 1e-6:
            shipments.append({"src": f, "dest": c, "tons": val, "lane": "focus_to_center", "unit_cost": get_cost(f,c,cost)})

ship_df = pd.DataFrame(shipments)
ship_df["lane_cost"] = ship_df["tons"] * ship_df["unit_cost"]

display(ship_df.sort_values(["lane","src","dest"]).head(25))

# Export outputs (optional but helpful for A1 / B1 evidence)
ship_df.to_csv(OUT_DIR / "solution_shipments.csv", index=False)
pd.DataFrame([{"status": LpStatus[prob.status], "objective_value": prob.objective.value()}]).to_json(
    OUT_DIR / "summary.json", orient="records", indent=2
)
print("Wrote:", OUT_DIR / "solution_shipments.csv")
print("Wrote:", OUT_DIR / "summary.json")


In [ ]:
# Checking constraints: hub capacity
TOL = 1e-6
for h in hubs:
    total_shipment = sum(x[h][f].varValue or 0 for f in focus_cities)
    cap = hubs[h]["capacity"]
    if total_shipment <= cap + TOL:
        print(f"Hub {h} capacity satisfied: {total_shipment} <= {cap}")
    else:
        print(f"Hub {h} capacity constraint violated: {total_shipment} > {cap}")


In [ ]:
# Checking constraints: focus city capacity
for f in focus_cities:
    total_processed = sum(y[f][c].varValue or 0 for c in flattened_centers)
    cap = focus_cities[f]["capacity"]
    if total_processed <= cap + TOL:
        print(f"Focus city {f} capacity satisfied: {total_processed} <= {cap}")
    else:
        print(f"Focus city {f} capacity constraint violated: {total_processed} > {cap}")


In [ ]:
# Checking constraints: center demand
for city, demand in flattened_centers.items():
    total_shipment_to_center = sum(y[f][city].varValue or 0 for f in focus_cities)
    if abs(total_shipment_to_center - demand) <= 1e-4:  # looser tolerance for float sums
        print(f"Center {city} demand satisfied: {total_shipment_to_center} == {demand}")
    else:
        print(f"Center {city} demand constraint violated: {total_shipment_to_center} != {demand}")


In [ ]:
# Checking constraints: flow balance for focus cities
for f in focus_cities:
    total_flow_in = sum(x[h][f].varValue or 0 for h in hubs)
    total_flow_out = sum(y[f][city].varValue or 0 for city in flattened_centers)
    if abs(total_flow_in - total_flow_out) <= 1e-4:
        print(f"Flow balance for focus city {f} satisfied: {total_flow_in} == {total_flow_out}")
    else:
        print(f"Flow balance for focus city {f} violated: {total_flow_in} != {total_flow_out}")


In [ ]:
# (Optional) Save a compact constraint-check report to CSV (nice evidence for B1)
rows = []
for h in hubs:
    lhs = sum(x[h][f].varValue or 0 for f in focus_cities)
    rhs = hubs[h]["capacity"]
    rows.append({"constraint":"HubCapacity", "entity":h, "lhs":lhs, "rhs":rhs, "sense":"<=", "satisfied": lhs <= rhs + TOL})

for f in focus_cities:
    lhs = sum(y[f][c].varValue or 0 for c in flattened_centers)
    rhs = focus_cities[f]["capacity"]
    rows.append({"constraint":"FocusCapacity", "entity":f, "lhs":lhs, "rhs":rhs, "sense":"<=", "satisfied": lhs <= rhs + TOL})

for c, demand in flattened_centers.items():
    lhs = sum(y[f][c].varValue or 0 for f in focus_cities)
    rhs = demand
    rows.append({"constraint":"CenterDemand", "entity":c, "lhs":lhs, "rhs":rhs, "sense":"==", "satisfied": abs(lhs-rhs) <= 1e-4})

for f in focus_cities:
    lhs = sum(x[h][f].varValue or 0 for h in hubs)
    rhs = sum(y[f][c].varValue or 0 for c in flattened_centers)
    rows.append({"constraint":"FlowBalance", "entity":f, "lhs":lhs, "rhs":rhs, "sense":"==", "satisfied": abs(lhs-rhs) <= 1e-4})

check_df = pd.DataFrame(rows)
check_df.to_csv(OUT_DIR / "constraint_checks.csv", index=False)
display(check_df.head(10))
print("Wrote:", OUT_DIR / "constraint_checks.csv")

# References (put these in your written narrative if needed):
# - PuLP documentation: https://coin-or.github.io/pulp/
# - CBC solver (COIN-OR): https://github.com/coin-or/Cbc
